# HeatUp — Notebook 1: Structure Preparation & AIMD

Runs the four pre-analysis workflows for each material:

| Step | What it does | Output |
|------|-------------|--------|
| Relaxation | BFGS geometry optimisation | `relaxation/CONTCAR`, `energy.json` |
| Phonons | Harmonic DOS (always) + QHA if configured | `phonons/dos.json` (+ `phonons/qha/` for QHA) |
| Elastic | 6×6 Voigt stiffness tensor | `elastic/elastic_tensor.json` |
| AIMD supercell | Build supercell, copy to each T sub-folder | `aimd/POSCAR`, `aimd/<T>K/POSCAR` |
| MD | NPT or NVT trajectory | `aimd/<T>K/output.traj`, `analysis.json` |
| VDOS | VACF → anharmonic phonon DOS and F(T) | `aimd/<T>K/anharmonic_phonons/vdos.json` |

Each step writes a `*_manifest.json` recording the exact configuration used.

**Idempotent**: re-running skips completed steps.  Set `FORCE_RERUN = True` to redo.

**Edit only the Configuration cell.**

In [1]:
import json, os, shutil, warnings
import torch
import heatup.config as cfg
from heatup.md_pipeline import (
    scan_database, print_database_summary, run_md_subprocess,
)
from heatup.structure_pipeline import (
    run_relaxation_subprocess,
    run_phonons_subprocess,
    run_elastic_subprocess,
    prepare_aimd_folders,
)
from heatup.anharmonic_phonons import compute_anharmonic_phonons_for_sim
from heatup.manifest import manifest_summary
warnings.filterwarnings('ignore')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device         : {DEVICE}')

Device         : cuda


## Configuration — **edit only this cell**

All parameters that affect the physics are set here.  They are written to
`*_manifest.json` files alongside every output so results remain traceable.

In [7]:
# ── Paths ──────────────────────────────────────────────────────────────────
DATABASE_ROOT  = 'database'
CANDIDATES_DIR = 'input/candidates'

# ── Calculator / model ─────────────────────────────────────────────────────
cfg.MACE_MODEL   = os.environ.get('MACE_MODEL_PATH', 'mace-mpa-0-medium.model')
cfg.CALC_BACKEND = 'mace-mp'   # 'mace-mp' | 'chgnet' | 'm3gnet' | 'custom'

# ── Phonon mode ─────────────────────────────────────────────────────────────
# Set the same value here AND in notebook 2 so the pipeline is consistent.
#
#   'HA'   — Harmonic approximation (fast, no AIMD required for F_vib).
#   'QHA'  — Quasi-harmonic (phonons at multiple volumes; needs phonopy).
#   'VDOS' — Anharmonic VDOS from AIMD (default, recommended for high-T).
#
# Regardless of PHONON_MODE, HA phonons are ALWAYS computed (cheap baseline
# + needed for competing-phase free energies in the thermodynamic hull).
cfg.PHONON_MODE = 'VDOS'          # 'HA' | 'QHA' | 'VDOS'

# Force-constant order (HA/QHA only).  2 = standard; 3 = phono3py (very slow).
cfg.FORCE_CONSTANT_ORDER = 2

# Backend for displacement calculations (HA/QHA only).
# 'ase' requires no extra deps.  'phonopy' required for QHA and IFC order=3.
cfg.PHONON_BACKEND = 'ase'

# ── QHA parameters (only used when PHONON_MODE = 'QHA') ────────────────────
cfg.QHA_N_VOLUMES    = 7      # odd integer ≥ 3
cfg.QHA_VOLUME_RANGE = 0.06   # ±6 % around V₀
cfg.QHA_EOS          = 'vinet'

# ── MD parameters ─────────────────────────────────────────────────────────
# Ensemble: 'NPT' (Martyna-Tobias-Klein, cell fluctuates) or
#           'NVT' (Nosé-Hoover, fixed cell, faster).
cfg.MD_ENSEMBLE    = 'NPT'
cfg.MD_TIMESTEP_FS = 1.0       # integration timestep (fs)
cfg.MD_N_STEPS     = 30_000    # total MD steps
cfg.MD_NBLOCK      = 20        # write one frame per NBLOCK steps
cfg.MD_STEP_EQUIV  = 100       # frames discarded as equilibration
cfg.MD_TTIME_FS    = 50.0      # Nosé-Hoover thermostat time constant (fs)
cfg.MD_PTIME_FS    = 500.0     # Parrinello-Rahman barostat time constant (fs, NPT only)
cfg.MD_PRESSURE_GPA = 0.0      # external pressure (GPa, NPT only)

# ── Relaxation parameters ──────────────────────────────────────────────────
cfg.RELAX_FMAX           = 0.05    # force convergence (eV/Å)
cfg.RELAX_MAX_STEPS      = 500
cfg.RELAX_CELL           = True    # also relax cell shape and volume
cfg.RELAX_CONSTANT_VOLUME = False  # fix volume when relaxing cell shape

# ── Phonon parameters (HA/QHA) ─────────────────────────────────────────────
cfg.PHONON_SUPERCELL = (3, 3, 3)  # supercell repetitions
cfg.PHONON_DELTA     = 0.05       # finite-displacement magnitude (Å)

# ── AIMD supercell ─────────────────────────────────────────────────────────
cfg.AIMD_MIN_CELL_ANG = 25.0   # minimum lattice parameter for supercell (Å)

# ── Temperature settings ───────────────────────────────────────────────────
HULL_TEMPERATURES = list(range(0, 1501, 50))   # for VDOS F(T) grid

# ── Run list ───────────────────────────────────────────────────────────────
USE_QUERY_FILE = False
QUERY_FILE     = 'next_candidates.json'

MANUAL_LIST = [
    {'material': 'Ba', 'symmetry': 'mp-1008283',  'temperature': 600},
    {'material': 'Ba', 'symmetry': 'mp-10679',  'temperature': 600},
    {'material': 'Ba', 'symmetry': 'mp-1096840',  'temperature': 600},
    {'material': 'Ba', 'symmetry': 'mp-1977763',  'temperature': 600},
    {'material': 'Ba', 'symmetry': 'mp-570384',  'temperature': 600},
    {'material': 'Ba', 'symmetry': 'mp-639747',  'temperature': 600},
    {'material': 'Ba', 'symmetry': 'mp-1058581',  'temperature': 600},
    {'material': 'Ba', 'symmetry': 'mp-1096835',  'temperature': 600},
    {'material': 'Ba', 'symmetry': 'mp-122',  'temperature': 600},
    {'material': 'Ba', 'symmetry': 'mp-56',  'temperature': 600},
    {'material': 'Ba', 'symmetry': 'mp-605790',  'temperature': 600},
    {'material': 'BaTiO3', 'symmetry': 'mp-1076932',  'temperature': 600},
    {'material': 'BaTiO3', 'symmetry': 'mp-2998',  'temperature': 600},
    {'material': 'BaTiO3', 'symmetry': 'mp-504715',  'temperature': 600},
    {'material': 'BaTiO3', 'symmetry': 'mp-5777',  'temperature': 600},
    {'material': 'BaTiO3', 'symmetry': 'mp-5986',  'temperature': 600},
    {'material': 'BaTiO3', 'symmetry': 'mp-995191',  'temperature': 600},
    {'material': 'BaTiO3', 'symmetry': 'mp-19990',  'temperature': 600},
    {'material': 'BaTiO3', 'symmetry': 'mp-5020',  'temperature': 600},
    {'material': 'BaTiO3', 'symmetry': 'mp-558125',  'temperature': 600},
    {'material': 'BaTiO3', 'symmetry': 'mp-5933',  'temperature': 600},
    {'material': 'BaTiO3', 'symmetry': 'mp-644497',  'temperature': 600},
    {'material': 'O', 'symmetry': 'mp-1009490',  'temperature': 600},
    {'material': 'O', 'symmetry': 'mp-1057818',  'temperature': 600},
    {'material': 'O', 'symmetry': 'mp-1087546',  'temperature': 600},
    {'material': 'O', 'symmetry': 'mp-1180050',  'temperature': 600},
    {'material': 'O', 'symmetry': 'mp-1524462',  'temperature': 600},
    {'material': 'O', 'symmetry': 'mp-2421172',  'temperature': 600},
    {'material': 'O', 'symmetry': 'mp-610917',  'temperature': 600},
    {'material': 'O', 'symmetry': 'mp-1023923',  'temperature': 600},
    {'material': 'O', 'symmetry': 'mp-1058623',  'temperature': 600},
    {'material': 'O', 'symmetry': 'mp-1102442',  'temperature': 600},
    {'material': 'O', 'symmetry': 'mp-1180064',  'temperature': 600},
    {'material': 'O', 'symmetry': 'mp-1525745',  'temperature': 600},
    {'material': 'O', 'symmetry': 'mp-2739215',  'temperature': 600},
    {'material': 'O', 'symmetry': 'mp-611836',  'temperature': 600},
    {'material': 'O', 'symmetry': 'mp-1056059',  'temperature': 600},
    {'material': 'O', 'symmetry': 'mp-1065697',  'temperature': 600},
    {'material': 'O', 'symmetry': 'mp-1180008',  'temperature': 600},
    {'material': 'O', 'symmetry': 'mp-12957',  'temperature': 600},
    {'material': 'O', 'symmetry': 'mp-2204849',  'temperature': 600},
    {'material': 'O', 'symmetry': 'mp-560602',  'temperature': 600},
    {'material': 'O', 'symmetry': 'mp-723285',  'temperature': 600},
    {'material': 'O', 'symmetry': 'mp-1056831',  'temperature': 600},
    {'material': 'O', 'symmetry': 'mp-1066100',  'temperature': 600},
    {'material': 'O', 'symmetry': 'mp-1180036',  'temperature': 600},
    {'material': 'O', 'symmetry': 'mp-1524452',  'temperature': 600},
    {'material': 'O', 'symmetry': 'mp-2206907',  'temperature': 600},
    {'material': 'O', 'symmetry': 'mp-607540',  'temperature': 600},
    {'material': 'O', 'symmetry': 'mp-734188',  'temperature': 600},
    {'material': 'Ti', 'symmetry': 'mp-1238818',  'temperature': 600},
    {'material': 'Ti', 'symmetry': 'mp-1245006',  'temperature': 600},
    {'material': 'Ti', 'symmetry': 'mp-1245170',  'temperature': 600},
    {'material': 'Ti', 'symmetry': 'mp-46',  'temperature': 600},
    {'material': 'Ti', 'symmetry': 'mp-72',  'temperature': 600},
    {'material': 'Ti', 'symmetry': 'mp-1244924',  'temperature': 600},
    {'material': 'Ti', 'symmetry': 'mp-1245164',  'temperature': 600},
    {'material': 'Ti', 'symmetry': 'mp-1245320',  'temperature': 600},
    {'material': 'Ti', 'symmetry': 'mp-6985',  'temperature': 600},
    {'material': 'Ti', 'symmetry': 'mp-73',  'temperature': 600},
]

# ── Step flags ─────────────────────────────────────────────────────────────
RUN_RELAXATION = True
RUN_PHONONS    = True   # always runs HA; also QHA if PHONON_MODE='QHA'
RUN_ELASTIC    = True
RUN_AIMD_PREP  = True
RUN_MD         = True
RUN_ANHARMONIC = True   # compute VDOS from trajectory (required for VDOS mode)
FORCE_RERUN    = False  # set True to recompute even if output exists

os.makedirs(DATABASE_ROOT, exist_ok=True)

_total_ps   = cfg.MD_N_STEPS * cfg.MD_TIMESTEP_FS / 1000.0
_equil_ps   = cfg.MD_STEP_EQUIV * cfg.MD_NBLOCK * cfg.MD_TIMESTEP_FS / 1000.0
_frame_fs   = cfg.MD_NBLOCK * cfg.MD_TIMESTEP_FS

print(f'Calculator     : {cfg.CALC_BACKEND}:{cfg.MACE_MODEL}')
print(f'PHONON_MODE    : {cfg.PHONON_MODE}  (IFC order={cfg.FORCE_CONSTANT_ORDER}, backend={cfg.PHONON_BACKEND})')
if cfg.PHONON_MODE == "QHA":
    print(f'QHA            : {cfg.QHA_N_VOLUMES} volumes, ±{cfg.QHA_VOLUME_RANGE*100:.0f}%, EOS={cfg.QHA_EOS}')
print(f'MD ensemble    : {cfg.MD_ENSEMBLE}')
print(f'MD length      : {_total_ps:.0f} ps  ({cfg.MD_N_STEPS} steps × {cfg.MD_TIMESTEP_FS} fs)')
print(f'Write rate     : {_frame_fs:.0f} fs/frame')
print(f'Equilibration  : {cfg.MD_STEP_EQUIV} frames = {_equil_ps:.1f} ps')
print(f'Relax fmax     : {cfg.RELAX_FMAX} eV/Å  relax_cell={cfg.RELAX_CELL}')
print(f'Min AIMD cell  : {cfg.AIMD_MIN_CELL_ANG} Å')
print(f'Database       : {os.path.abspath(DATABASE_ROOT)}')

Calculator     : mace-mp:mace-mpa-0-medium.model
PHONON_MODE    : VDOS  (IFC order=2, backend=ase)
MD ensemble    : NPT
MD length      : 30 ps  (30000 steps × 1.0 fs)
Write rate     : 20 fs/frame
Equilibration  : 100 frames = 2.0 ps
Relax fmax     : 0.05 eV/Å  relax_cell=True
Min AIMD cell  : 25.0 Å
Database       : /home/claudio/cibran/Work/UPC/HeatUp/database


## Build run list

In [8]:
if USE_QUERY_FILE:
    if not os.path.exists(QUERY_FILE):
        raise FileNotFoundError(f'{QUERY_FILE} not found.')
    with open(QUERY_FILE) as fh:
        run_list = json.load(fh)
else:
    run_list = MANUAL_LIST

resolved = []
for entry in run_list:
    mat, sym, temp = entry['material'], entry['symmetry'], int(entry['temperature'])
    sym_dir     = os.path.join(DATABASE_ROOT, mat, sym)
    poscar_db   = os.path.join(sym_dir, 'POSCAR')
    poscar_cand = os.path.join(CANDIDATES_DIR, mat, sym, 'POSCAR')

    if os.path.exists(poscar_db):
        src = poscar_db
    elif os.path.exists(poscar_cand):
        src = poscar_cand
    else:
        print(f'  [MISSING] {mat}/{sym}: POSCAR not found'); continue

    os.makedirs(sym_dir, exist_ok=True)
    dst = os.path.join(sym_dir, 'POSCAR')
    if not os.path.exists(dst):
        shutil.copy(src, dst)
        print(f'  POSCAR copied → {dst}')

    resolved.append({
        'material'   : mat,
        'symmetry'   : sym,
        'temperature': temp,
        'sym_dir'    : sym_dir,
        'force_rerun': entry.get('force_rerun', FORCE_RERUN),
    })

print(f'Resolved {len(resolved)} entries:')
for e in resolved:
    print(f"  {e['material']}/{e['symmetry']}  T={e['temperature']} K")

  [MISSING] Ti/mp-1238818: POSCAR not found
  [MISSING] Ti/mp-1244924: POSCAR not found
Resolved 58 entries:
  Ba/mp-1008283  T=600 K
  Ba/mp-10679  T=600 K
  Ba/mp-1096840  T=600 K
  Ba/mp-1977763  T=600 K
  Ba/mp-570384  T=600 K
  Ba/mp-639747  T=600 K
  Ba/mp-1058581  T=600 K
  Ba/mp-1096835  T=600 K
  Ba/mp-122  T=600 K
  Ba/mp-56  T=600 K
  Ba/mp-605790  T=600 K
  BaTiO3/mp-1076932  T=600 K
  BaTiO3/mp-2998  T=600 K
  BaTiO3/mp-504715  T=600 K
  BaTiO3/mp-5777  T=600 K
  BaTiO3/mp-5986  T=600 K
  BaTiO3/mp-995191  T=600 K
  BaTiO3/mp-19990  T=600 K
  BaTiO3/mp-5020  T=600 K
  BaTiO3/mp-558125  T=600 K
  BaTiO3/mp-5933  T=600 K
  BaTiO3/mp-644497  T=600 K
  O/mp-1009490  T=600 K
  O/mp-1057818  T=600 K
  O/mp-1087546  T=600 K
  O/mp-1180050  T=600 K
  O/mp-1524462  T=600 K
  O/mp-2421172  T=600 K
  O/mp-610917  T=600 K
  O/mp-1023923  T=600 K
  O/mp-1058623  T=600 K
  O/mp-1102442  T=600 K
  O/mp-1180064  T=600 K
  O/mp-1525745  T=600 K
  O/mp-2739215  T=600 K
  O/mp-611836  T=600 

In [9]:
print_database_summary(DATABASE_ROOT)

No completed simulations in database


## Run structure preparation and AIMD

Each step is isolated in a subprocess so CUDA memory is fully released
between materials.  Failed steps print `[WARN]` but do not abort the loop.

In [ ]:
n_ok = n_fail = 0

for entry in resolved:
    mat, sym, temp = entry['material'], entry['symmetry'], entry['temperature']
    sd, force      = entry['sym_dir'], entry['force_rerun']
    tag = f'{mat}/{sym}  T={temp} K'
    print(f'\n{"="*65}\n  {tag}\n{"="*65}')

    # ── Gate 0: Relaxation ───────────────────────────────────────────────
    if RUN_RELAXATION:
        if not run_relaxation_subprocess(sd, device=DEVICE, force_rerun=force):
            print(f'  [WARN] Relaxation failed — downstream steps may be skipped.')

    # ── Phonons (HA always; QHA if configured) ───────────────────────────
    # run_phonons_subprocess respects cfg.PHONON_MODE:
    #   HA   → writes phonons/dos.json
    #   QHA  → writes phonons/dos.json + phonons/qha/qha_result.json
    #   VDOS → writes phonons/dos.json only (VDOS computed after MD below)
    if RUN_PHONONS:
        if not run_phonons_subprocess(sd, device=DEVICE, force_rerun=force):
            print(f'  [WARN] Phonons failed.')

    # ── Elastic tensor ───────────────────────────────────────────────────
    if RUN_ELASTIC:
        if not run_elastic_subprocess(sd, device=DEVICE, force_rerun=force):
            print(f'  [WARN] Elastic tensor failed.')

    # ── AIMD supercell ───────────────────────────────────────────────────
    if RUN_AIMD_PREP:
        try:
            prepare_aimd_folders(sd, temperatures=[temp], force_rebuild=force)
        except Exception as exc:
            print(f'  [WARN] AIMD prep: {exc}')
            continue

    # ── MD trajectory ────────────────────────────────────────────────────
    # Ensemble (NPT/NVT), timestep, length etc. all from cfg above.
    if RUN_MD:
        if not run_md_subprocess(sd, temp, device=DEVICE, force_rerun=force):
            print(f'  [FAIL] MD'); n_fail += 1; continue

    # ── Anharmonic VDOS ──────────────────────────────────────────────────
    # Required when PHONON_MODE = 'VDOS'; always computed when RUN_ANHARMONIC
    # is True so the stability gate (Gate 2) has VDOS data regardless of mode.
    if RUN_ANHARMONIC:
        sim_dir = os.path.join(sd, 'aimd', f'{temp}K')
        if os.path.isdir(sim_dir):
            try:
                fe = compute_anharmonic_phonons_for_sim(
                    sim_dir,
                    temperatures=HULL_TEMPERATURES,
                    force_recompute=force,
                )
                if fe is not None:
                    print(f'  VDOS → {sim_dir}/anharmonic_phonons/')
            except Exception as exc:
                print(f'  [WARN] VDOS: {exc}')

    n_ok += 1

print(f'\nDone: {n_ok} completed, {n_fail} failed.')


  Ba/mp-1008283  T=600 K


/home/claudio/cibran/Work/UPC/HeatUp/.venv/lib/python3.12/site-packages/e3nn/o3/_wigner.py:10: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  _Jd, _W3j_flat, _W3j_indices = torch.load(os.path.join(os.path.dirname(__file__), 'constants.pt'))
/home/claudio/cibran/Work/UPC/HeatUp/.venv/lib/python3.12/site-packages/mace/calculators/mace.py:226: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  torch.load(f=model_path, map_location=device)
/home/claudio/cibran/Work/UPC/HeatUp/heatup/structure_pipeline.py:451: DeprecationWarning: Use FrechetCellFilter for better convergence w.r.t. cell variables.
  ExpCellFilter(atoms, constant_volume=config.RELAX_CONSTANT_VOLUME)


  Space group: 191
cuequivariance or cuequivariance_torch is not available. Cuequivariance acceleration will be disabled.
Using float64 for MACECalculator, which is slower but more accurate. Recommended for geometry optimization.
Relaxing Ba/mp-1008283  (fmax=0.05 eV/Å, relax_cell=True, backend=mace-mp:mace-mpa-0-medium.model) ...
  Relaxation done.  E0 = -1.215418 eV/atom → database/Ba/mp-1008283/relaxation/CONTCAR


/home/claudio/cibran/Work/UPC/HeatUp/.venv/lib/python3.12/site-packages/e3nn/o3/_wigner.py:10: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  _Jd, _W3j_flat, _W3j_indices = torch.load(os.path.join(os.path.dirname(__file__), 'constants.pt'))
/home/claudio/cibran/Work/UPC/HeatUp/.venv/lib/python3.12/site-packages/mace/calculators/mace.py:226: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  torch.load(f=model_path, map_location=device)


HA phonons for Ba/mp-1008283  (backend=ase, IFC order=2, supercell=(3, 3, 3), δ=0.05 Å) ...
cuequivariance or cuequivariance_torch is not available. Cuequivariance acceleration will be disabled.
Using float64 for MACECalculator, which is slower but more accurate. Recommended for geometry optimization.
WARNING, 3 imaginary frequencies at q = ( 0.00,  0.00,  0.00) ; (omega_q = 5.594e-02*i)
WARNING, 2 imaginary frequencies at q = ( 0.06,  0.00,  0.00) ; (omega_q = 5.525e-02*i)
WARNING, 2 imaginary frequencies at q = ( 0.11,  0.00,  0.00) ; (omega_q = 5.321e-02*i)
WARNING, 2 imaginary frequencies at q = ( 0.17,  0.00,  0.00) ; (omega_q = 4.985e-02*i)
WARNING, 2 imaginary frequencies at q = ( 0.22,  0.00,  0.00) ; (omega_q = 4.528e-02*i)
WARNING, 2 imaginary frequencies at q = ( 0.28,  0.00,  0.00) ; (omega_q = 3.963e-02*i)
WARNING, 2 imaginary frequencies at q = ( 0.33,  0.00,  0.00) ; (omega_q = 3.307e-02*i)
WARNING, 2 imaginary frequencies at q = ( 0.39,  0.00,  0.00) ; (omega_q = 3.124e

/home/claudio/cibran/Work/UPC/HeatUp/heatup/phonons.py:812: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout(pad=0.5)



WARNING, 1 imaginary frequencies at q = ( 0.45, -0.15, -0.45) ; (omega_q = 5.829e-02*i)
WARNING, 1 imaginary frequencies at q = ( 0.45, -0.15, -0.35) ; (omega_q = 5.532e-02*i)
WARNING, 1 imaginary frequencies at q = ( 0.45, -0.15, -0.25) ; (omega_q = 5.013e-02*i)
WARNING, 1 imaginary frequencies at q = ( 0.45, -0.15, -0.15) ; (omega_q = 4.433e-02*i)
WARNING, 1 imaginary frequencies at q = ( 0.45, -0.15, -0.05) ; (omega_q = 4.020e-02*i)
WARNING, 1 imaginary frequencies at q = ( 0.45, -0.15,  0.05) ; (omega_q = 4.020e-02*i)
WARNING, 1 imaginary frequencies at q = ( 0.45, -0.15,  0.15) ; (omega_q = 4.433e-02*i)
WARNING, 1 imaginary frequencies at q = ( 0.45, -0.15,  0.25) ; (omega_q = 5.013e-02*i)
WARNING, 1 imaginary frequencies at q = ( 0.45, -0.15,  0.35) ; (omega_q = 5.532e-02*i)
WARNING, 1 imaginary frequencies at q = ( 0.45, -0.15,  0.45) ; (omega_q = 5.829e-02*i)
WARNING, 1 imaginary frequencies at q = ( 0.45, -0.05, -0.45) ; (omega_q = 5.629e-02*i)
WARNING, 1 imaginary frequencie

/home/claudio/cibran/Work/UPC/HeatUp/.venv/lib/python3.12/site-packages/e3nn/o3/_wigner.py:10: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  _Jd, _W3j_flat, _W3j_indices = torch.load(os.path.join(os.path.dirname(__file__), 'constants.pt'))
/home/claudio/cibran/Work/UPC/HeatUp/.venv/lib/python3.12/site-packages/mace/calculators/mace.py:226: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  torch.load(f=model_path, map_location=device)


cuequivariance or cuequivariance_torch is not available. Cuequivariance acceleration will be disabled.
Using float64 for MACECalculator, which is slower but more accurate. Recommended for geometry optimization.
Computing elastic tensor for Ba/mp-1008283  (delta=±0.0001, 12 single-points, backend=mace-mp:mace-mpa-0-medium.model) ...
  Elastic done.  B=3.6  G=6.0  E=11.7 GPa  ν=-0.034
AIMD supercell for Ba/mp-1008283:  reps=(4, 4, 7)  unit=[6.98 6.98 4.  ] Å  sc=[27.94 27.94 27.98] Å  atoms=224
  Master supercell → database/Ba/mp-1008283/aimd/POSCAR
  [skip] database/Ba/mp-1008283/aimd/600K/POSCAR already exists.


/home/claudio/cibran/Work/UPC/HeatUp/.venv/lib/python3.12/site-packages/e3nn/o3/_wigner.py:10: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  _Jd, _W3j_flat, _W3j_indices = torch.load(os.path.join(os.path.dirname(__file__), 'constants.pt'))
/home/claudio/cibran/Work/UPC/HeatUp/.venv/lib/python3.12/site-packages/mace/calculators/mace.py:226: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  torch.load(f=model_path, map_location=device)
/home/claudio/cibran/Work/UPC/HeatUp/heatup/md_pipeline.py:583: FutureWarning: NPT thermostat has been moved/renamed to ase.md.melchionna.MelchionnaNPT. Please use this class instead (or a newer NPT thermostat!)
  dyn = NPT(atoms,


## Database summary

In [ ]:
print_database_summary(DATABASE_ROOT)

## Manifest check

Prints the manifest for the first completed simulation — confirms the exact
configuration written alongside every output file.

In [ ]:
import glob

manifests = sorted(glob.glob(f'{DATABASE_ROOT}/**/*_manifest.json', recursive=True))
if manifests:
    print(f'Found {len(manifests)} manifest(s).  First:')
    print(manifest_summary(manifests[0]))
else:
    print('No manifests found yet — run the pipeline above.')